<a href="https://colab.research.google.com/github/20230532/maritime-data-mining/blob/main/11_week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# [실습 1] channel_purchase.csv : 카이제곱 독립성 검정

from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency

# 경로 설정
DATA_DIR = Path('/content/drive/MyDrive/type3_week')
file_path = DATA_DIR / 'channel_purchase.csv'

# 1. 데이터 불러오기
df = pd.read_csv(file_path)

# 2. 데이터 확인
print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[변수 정보]')
print(df.info(), '\n')

# 3. 교차표 생성
table = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(table, '\n')

# 4. 카이제곱 독립성 검정
chi2, p, dof, expected = chi2_contingency(table)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

# 5. 기대도수 확인
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)

print('\n[기대도수]')
print(expected_df)

# 6. 채널별 구매전환율
conversion_rate = df.groupby('channel')['purchase_yn'].mean().sort_values(ascending=False) * 100

print('\n[채널별 구매전환율(%)]')
print(conversion_rate.round(2))

# 7. 해석 출력
print('\n[해석]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

print(f'가장 높은 구매전환율 채널은 {conversion_rate.index[0]}이며, 전환율은 {conversion_rate.iloc[0]:.2f}%입니다.')

[데이터 상위 5행]
  channel  purchase_yn
0    검색광고            1
1    검색광고            1
2    검색광고            1
3    검색광고            1
4    검색광고            1 

[변수 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   channel      530 non-null    object
 1   purchase_yn  530 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 8.4+ KB
None 

[교차표]
purchase_yn   0    1
channel             
SNS          55   70
검색광고         50   90
이메일          80   45
직접방문         30  110 

[검정 결과]
chi-square statistic: 51.716
p-value: 0.0
degrees of freedom: 3

[기대도수]
purchase_yn          0          1
channel                          
SNS          50.707547  74.292453
검색광고         56.792453  83.207547
이메일          50.707547  74.292453
직접방문         56.792453  83.207547

[채널별 구매전환율(%)]
channel
직접방문    78.57
검색광고    64.29
SNS     56.00
이메일     36.00
Name: purchase_yn

In [9]:
# [실습 1] retail_sales_reg1.csv : 다중회귀 01

from google.colab import drive
from pathlib import Path
import pandas as pd
import statsmodels.formula.api as smf

# 1. 드라이브 마운트
# drive.mount('/content/drive')

# 2. 경로 설정
DATA_DIR = Path('/content/drive/MyDrive/type3_week')
file_path = DATA_DIR / 'retail_sales_reg1.csv'

# 3. 데이터 불러오기
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[기본 정보]')
print(df.info(), '\n')

# 4. 회귀모형 적합, Ordinary Least Squares(최소자승법)
model = smf.ols('sales ~ ad_cost + staff + event_cnt', data=df).fit()

# 5. 결과 요약
print('[회귀 요약 결과]')
print(model.summary())

# 6. 핵심 결과 추출
print('\n[회귀계수]')
print(model.params)

print('\n[p-value]')
print(model.pvalues)

print('\n[R-squared]')
print(model.rsquared)

# 7. 광고비 10단위 증가 효과
beta_ad = model.params['ad_cost']
print('\n광고비 10단위 증가 시 예상 매출 변화:', beta_ad * 10)

# 8. 유의한 변수 목록
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print('\n유의한 변수:', sig_vars)

[데이터 상위 5행]
    sales  ad_cost  staff  event_cnt
0  344.42    38.61     12          4
1  316.86    49.40     14          3
2  290.08    49.31      5          4
3  356.27    46.20     15          4
4  273.46    50.36     13          2 

[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sales      80 non-null     float64
 1   ad_cost    80 non-null     float64
 2   staff      80 non-null     int64  
 3   event_cnt  80 non-null     int64  
dtypes: float64(2), int64(2)
memory usage: 2.6 KB
None 

[회귀 요약 결과]
                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.881
Method:                 Least Squares   F-statistic:                     196.5
Date:                Mon, 01 J

In [10]:
# [실습 1] churn_logit1.csv 로지스틱 회귀: 이탈 여부와 오즈비 해석

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/type3_week')  # 필요시 수정
df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수표 정리
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})

print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 95% 신뢰구간과 오즈비 기준으로도 보기
conf = model.conf_int() #interval
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])   #odd비는 exp
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 예측확률 및 분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #결과를 정수로

print("\n=== 예측확률 상위 5개 ===")
print(df[['churn', 'pred_prob', 'pred_class']].head())

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.4f}, p-value={pval:.6f}, 오즈비={or_val:.4f}")
    print(f" -> {var}가 1단위 증가할 때 이탈 odds는 약 {or_val:.4f}배가 되며, 방향은 {direction}입니다.")


=== 데이터 미리보기 ===
   churn  age  usage_hour  complaint_cnt
0      0   29       12.07              0
1      0   20       70.05              2
2      0   23       58.09              0
3      0   37       67.25              2
4      0   54       78.19              0

=== 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   churn          500 non-null    int64  
 1   age            500 non-null    int64  
 2   usage_hour     500 non-null    float64
 3   complaint_cnt  500 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 15.8 KB
None
Optimization terminated successfully.
         Current function value: 0.237853
         Iterations 7

=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  500
Model:            